# ST-Reason · 07 QLoRA (optional, H5)

**Question (H5):** does a small QLoRA fine-tune teach Qwen3-4B to reason over our video memories, and does it narrow the gap to the untuned Qwen3-8B?

**Training data, without any new CV runs:**

| Source | Examples | Memory |
| --- | --- | --- |
| STAR **train** split (Sequence and Interaction questions; videos never used in our STAR test sets) | `N_STAR_TRAIN` (default 600) | ORACLE_M4, built from STAR's ground truth exactly as notebook 04 builds it for the oracle runs |
| NExT-QA **dev** set (already used for the controls in 05) | all with a memory (about 300) | the real CV memory M4 from notebook 04 |

The STAR part teaches the format and the reasoning over clean facts; the dev part adds the noise of real CV memories. No test question (main grid, STAR test sets) is ever used for training.

**Target** for each example: the reply format of strategy A, `{"answer": "<letter>", "evidence": [...]}`. The evidence IDs are the memory items that share the most words with the correct option (an automatic heuristic, no labelling).

**Training:** 4-bit NF4 base, LoRA rank 16 on all attention and MLP projections, 1 epoch, learning rate 2e-4, effective batch 8, fp16 on the T4. The loss is on the reply only.

**Evaluation (strategy A, the test shards of 05):**

| Run | Model |
| --- | --- |
| `main.M4.A.q4b`, `star.M4.A.q4b` | Qwen3-4B, untuned |
| `main.M4.A.q4b-lora`, `star.M4.A.q4b-lora` | Qwen3-4B + this adapter |
| `star.M4.A` (reference) | Qwen3-8B, untuned (`main.M4.A` already exists from 05) |

**Order of work (each step is skipped once done, so after a disconnect just Run all):** build the training set → answer with the untuned 4B → train (checkpointed to Drive every 20 minutes) → answer with the tuned 4B → answer with the 8B reference.

**Runtime:** GPU (T4). About 2–2.5 hours in total, so 2 sessions. The main cell stops cleanly after 110 minutes; Run all again to continue.

In [1]:
# --- Parameters: the only cell to edit -----------------------------------------------------------
EVAL_SHARDS = [0, 1]      # the question shards answered in 05
N_SHARDS = 4
N_STAR_TRAIN = 600        # STAR train questions used for training
USE_DEV = True            # also train on the NExT-QA dev questions with their real CV memory
MAX_LEN = 1536            # tokens per training example (memory is shortened to fit)
LORA_R = 16
LR = 2e-4
EPOCHS = 1
GRAD_ACCUM = 8
SAVE_EVERY_MIN = 20       # checkpoint the training to Drive at least this often
DO_8B_REFERENCE = True    # also answer star.M4.A with the untuned Qwen3-8B
LIMIT_QUESTIONS = None    # e.g. 8 for a quick test (per eval run); None for the real run
SYNC_EVERY_MIN = 8

In [2]:
# --- 0. Bootstrap --------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '07_qlora'
ST_QUIET = True
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

MOCK = os.environ.get('ST_MOCK_MODELS') == '1'
SHARD = min(EVAL_SHARDS)                                    # used in file names only
if not MOCK:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('No GPU in this runtime. Runtime -> Change runtime type -> T4 GPU -> Save, then Run all. '
                           'If T4 cannot be chosen, this Google account has no free GPU right now (its daily quota is '
                           'used, or another notebook of this account already has the GPU): use another account or try later.')
MANIFEST_PATH = P['runs'] / f'manifest_07.csv'
DRIVE_LOG_PATH = LOG_PATH
LOG_PATH = P['local'] / 'logs' / LOG_PATH.name
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
import pandas as pd, numpy as np, pickle, types
# outputs/qlora/ if it exists on Drive (files only, no new sub-folders: folders made from Colab inside a shared
# Drive folder can vanish); otherwise outputs/predictions/, which certainly exists
QL = P['qlora'] if P['qlora'].exists() else P['pred']
QL_LOCAL = P['local'] / 'qlora'
QL_LOCAL.mkdir(parents=True, exist_ok=True)
STAR = P['data'] / 'star'

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87


## 1. Memories and test questions

The newest memory zip from 04 is unpacked as in 05, and the test questions are the same frozen sets with the same shard rule.

In [3]:
MEM_LOCAL = P['local'] / 'memory_in'
packs = sorted(P['memory'].glob('04_memories_*.zip'))
assert packs, 'no memory zip in outputs/memory: run notebook 04 first'
MEM_PACK = packs[-1].name
if load_json(MEM_LOCAL / 'source.json', default={}).get('pack') != MEM_PACK:
    shutil.rmtree(MEM_LOCAL, ignore_errors=True)
    MEM_LOCAL.mkdir(parents=True)
    local_zip = P['local_tmp'] / MEM_PACK
    shutil.copyfile(packs[-1], local_zip)
    with zipfile.ZipFile(local_zip) as zf:
        zf.extractall(MEM_LOCAL)
    save_json({'pack': MEM_PACK}, MEM_LOCAL / 'source.json')
print('memories from', MEM_PACK, '| formats:', sorted(p.name for p in MEM_LOCAL.iterdir() if p.is_dir()))

_MEM = {}
def memory(fmt, key):
    k = (fmt, key)
    if k not in _MEM:
        _MEM[k] = load_json(MEM_LOCAL / fmt / f'{key}.json')
    return _MEM[k]

def read_set(name):
    return pd.read_csv(P['subsets'] / f'{name}.csv', dtype={'video_id': str})
def shard_of(uid):
    return int(hashlib.sha1(str(uid).encode()).hexdigest(), 16) % N_SHARDS

def nextqa_questions(df):
    return [{'uid': r.uid, 'key': str(r.video_id), 'question': r.question.rstrip('?') + '?',
             'options': [str(getattr(r, f'a{i}')) for i in range(5)], 'gold': int(r.answer_idx),
             'type': r.type} for r in df.itertuples(index=False)]
def star_questions(df, version='orig'):
    out = []
    for r in df.itertuples(index=False):
        n = int(getattr(r, 'n_options', 4) or 4)
        out.append({'uid': r.uid if version == 'orig' else f'{r.uid}__{version}', 'base_uid': r.uid,
                    'key': f'{r.uid}__{version}', 'question': str(r.question),
                    'options': [str(getattr(r, f'a{i}')).rstrip('.') for i in range(n)],
                    'gold': int(r.answer_idx), 'type': r.type, 'version': version})
    return out

QSETS = {'main': nextqa_questions(read_set('main_grid_eval')),
         'dev': nextqa_questions(read_set('dev')),
         'star': star_questions(read_set('star_main'))}

for name, qs in QSETS.items():
    for q in qs:
        q['shard'] = shard_of(q.get('base_uid', q['uid']))
EVAL_Q = {n: [q for q in QSETS[n] if q['shard'] in EVAL_SHARDS] for n in ['main', 'star']}
if LIMIT_QUESTIONS:
    EVAL_Q = {n: qs[:LIMIT_QUESTIONS] for n, qs in EVAL_Q.items()}
print({n: len(qs) for n, qs in EVAL_Q.items()}, 'test questions in shards', EVAL_SHARDS)

memories from 04_memories_20261017-124338.zip | formats: ['CAP', 'M1', 'M2', 'M3', 'M4', 'ORACLE_M1', 'ORACLE_M2', 'ORACLE_M3', 'ORACLE_M4', 'VA']
{'main': 193, 'star': 190} test questions in shards [0, 1]


## 2. Saving answers (verified zips)

As in 05 and 06: answers are written to the Colab disk and uploaded to `outputs/predictions/07_shard<k>_….zip`, read back to verify.

In [4]:
LOCAL_PRED = P['local'] / 'pred_out_07'
LOCAL_PRED.mkdir(parents=True, exist_ok=True)
PRED_DRIVE = P['pred']
_PACKED = set()

class DriveSyncError(BaseException):
    pass

def restore_preds():
    n = 0
    for z in sorted(PRED_DRIVE.glob('07_*.zip'), reverse=True):
        try:
            with zipfile.ZipFile(z) as zf:
                for m in zf.namelist():
                    _PACKED.add(m)
                    out = LOCAL_PRED / m
                    if not out.exists():
                        out.parent.mkdir(parents=True, exist_ok=True)
                        with zf.open(m) as s, open(out, 'wb') as d:
                            shutil.copyfileobj(s, d)
                        n += 1
        except (zipfile.BadZipFile, OSError):
            log(f'damaged zip skipped: {z.name}', 'WARN')
    log(f'restored {n} answer files from Drive')

def sync_preds():
    new = [str(p.relative_to(LOCAL_PRED)) for p in LOCAL_PRED.rglob('*.jsonl') if str(p.relative_to(LOCAL_PRED)) not in _PACKED]
    if not new:
        return 0
    name = f'07_shard{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S}_{len(new):05d}.zip'
    tmp = P['local_tmp'] / name
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for rel in new:
            zf.write(LOCAL_PRED / rel, rel)
    ok = False
    for attempt in range(3):
        try:
            shutil.copyfile(tmp, PRED_DRIVE / name)
            with zipfile.ZipFile(PRED_DRIVE / name) as zf:
                ok = len(zf.namelist()) == len(new) and zf.testzip() is None
            if ok:
                break
        except Exception as e:
            log(f'upload of {name} failed ({e.__class__.__name__}); retrying', 'WARN')
        time.sleep(5 * (attempt + 1))
    if not ok:
        raise DriveSyncError(f'Drive did not keep {name}. Answers are still on this Colab disk: do NOT restart; '
                             'run the summary cell (section 7) to retry the upload.')
    _PACKED.update(new)
    tmp.unlink(missing_ok=True)
    try:
        shutil.copyfile(LOG_PATH, DRIVE_LOG_PATH)
    except Exception:
        pass
    log(f'uploaded {name}: {len(new)} files, verified on Drive')
    return len(new)

def answered(run_id):
    out = set()
    for f in (LOCAL_PRED / run_id).glob('*.jsonl'):
        out |= {r['uid'] for r in read_jsonl(f)}
    return out

restore_preds()

def put_on_drive(local_path, name):
    """Copy one file into outputs/qlora/ and read it back (zip test) to prove it arrived."""
    tmp = QL / (name + '.part')
    for attempt in range(3):
        try:
            shutil.copyfile(local_path, tmp)
            os.replace(tmp, QL / name)
            if str(name).endswith('.zip'):
                with zipfile.ZipFile(QL / name) as zf:
                    assert zf.testzip() is None
            return
        except Exception as e:
            log(f'upload of {name} failed ({e.__class__.__name__}); retrying', 'WARN')
            time.sleep(5 * (attempt + 1))
    raise DriveSyncError(f'Drive did not keep {name}: do NOT restart; run this cell again.')

[14:33:26] INFO  restored 9 answer files from Drive


## 3. Training set

**STAR train.** `STAR_train.json` is downloaded from the STAR authors' storage (or taken from `data/source/star/` if uploaded by hand) and read as a stream, so the whole file never has to fit in memory. Sequence and Interaction questions whose video is not in any of our STAR test sets are candidates; a fixed-seed sample of them is kept (at most 2 per video). Each becomes an ORACLE_M4 memory with exactly the code of notebooks 01b and 04.

**NExT-QA dev.** Every dev question with an M4 memory from 04.

The finished set is saved once as `outputs/qlora/07_train_set.jsonl` and reused after that, so every session trains on the same examples.

In [5]:
TRAIN_SET = QL / '07_train_set.jsonl'
S3 = 'https://star-benchmark.s3.us-east.cloud-object-storage.appdomain.cloud'
STAR_TRAIN_URL = (CFG.get('star', {}).get('urls', {}) or {}).get('STAR_train.json',
                                                                  f'{S3}/Question_Answer_SituationGraph/STAR_train.json')

def read_classes(path):
    out = {}
    for line in open(path):
        parts = line.strip().split(' ', 1)
        if len(parts) == 2:
            out[parts[0]] = parts[1]
    return out

ACTION_NAME = read_classes(STAR / 'raw' / 'action_classes.txt')
VERB = read_classes(STAR / 'raw' / 'verb_classes.txt')
OBJ = read_classes(STAR / 'raw' / 'object_classes.txt')
REL = read_classes(STAR / 'raw' / 'relationship_classes.txt')
AMAP = {a: tuple(v.split()) for a, v in read_classes(STAR / 'raw' / 'action_mapping.txt').items()}
ACTION_VO = {a: (VERB[v], OBJ[o]) for a, (v, o) in AMAP.items()}          # 'a002' -> ('take', 'clothes')
VO_ACTION = {vo: a for a, vo in ACTION_VO.items()}
FPS = pickle.load(open(STAR / 'raw' / 'video_fps', 'rb'))                 # 'BE43L.mp4' -> 29.97
VERB_WORDS, OBJ_WORDS = set(VERB.values()), set(OBJ.values())

# Charades class names -> STAR action ids. STAR names are Charades names with base-form verbs,
# so the match is exact after lemmatising; a fuzzy match handles anything left over.
import itertools
IRREG = {'putting': 'put', 'sitting': 'sit', 'lying': 'lie', 'taking': 'take', 'closing': 'close',
         'tidying': 'tidy', 'throwing': 'throw', 'holding': 'hold', 'washing': 'wash', 'opening': 'open',
         'standing': 'stand', 'eating': 'eat', 'drinking': 'drink', 'watching': 'watch', 'reading': 'read',
         'looking': 'look', 'reaching': 'reach', 'grabbing': 'grab', 'walking': 'walk', 'turning': 'turn',
         'grasping': 'grasp', 'consuming': 'consume'}
def lemma_word(w):
    if w in IRREG:
        return [IRREG[w]]
    if w.endswith('ing') and len(w) > 5:
        b = w[:-3]; out = [b, b + 'e']
        if len(b) > 2 and b[-1] == b[-2]:
            out.append(b[:-1])
        return out
    return [w]
def lemma_variants(name):
    variants = ['']
    for tok in name.lower().split():
        forms = ['/'.join(p) for p in itertools.product(*[lemma_word(x) for x in tok.split('/')])]
        variants = [f'{v} {f}'.strip() for v in variants for f in forms][:256]
    return variants
def norm(s):
    return re.sub(r'\s+', ' ', str(s).lower().strip())
def stems(s):
    out = set()
    for w in re.findall(r'[a-z]+', s.lower()):
        for suf in ('ing', 'es', 's', 'ed'):
            if w.endswith(suf) and len(w) > len(suf) + 2:
                w = w[: -len(suf)]; break
        out.add(w)
    return out - {'some', 'something', 'somewhere', 'a', 'the', 'from', 'on', 'in', 'into', 'with', 'of', 'at', 'to'}

CHA_CLASSES = read_classes(STAR / 'raw' / 'Charades_v1_classes.txt')
STAR_BY_NAME = {norm(n): a for a, n in ACTION_NAME.items()}
STAR2CHA, how = {}, {}
for c, cname in CHA_CLASSES.items():
    for v in lemma_variants(cname):
        if v in STAR_BY_NAME and STAR_BY_NAME[v] not in STAR2CHA:
            STAR2CHA[STAR_BY_NAME[v]] = c; how[STAR_BY_NAME[v]] = 'exact'; break
for a, name in ACTION_NAME.items():
    if a in STAR2CHA:
        continue
    best, score = None, 0.0
    for c, cname in CHA_CLASSES.items():
        sa, sc = stems(name), stems(cname); j = len(sa & sc) / max(1, len(sa | sc))
        if j > score:
            best, score = c, j
    if score >= 0.5:
        STAR2CHA[a] = best; how[a] = 'fuzzy'
n_exact = sum(v == 'exact' for v in how.values())
print(f'STAR actions mapped to Charades classes: {len(STAR2CHA)}/{len(ACTION_NAME)} '
      f'({n_exact} exact, {len(STAR2CHA) - n_exact} fuzzy)')
unmapped = [ACTION_NAME[a] for a in ACTION_NAME if a not in STAR2CHA]
if unmapped:
    print('  unmapped (times fall back to keyframes):', unmapped)

CHA_ACTIONS = {}                                                        # video -> list of (class, start, end)
for split in ['train', 'test']:
    df = pd.read_csv(STAR / 'raw' / f'Charades_v1_{split}.csv', dtype=str)
    for vid, acts in zip(df['id'], df['actions'].fillna('')):
        CHA_ACTIONS[vid] = [(c, float(s), float(e)) for c, s, e in
                            (a.split() for a in acts.split(';') if len(a.split()) == 3)]

CHA_NAME = CHA_CLASSES
def oracle_record(q, r):
    vid, cs, ce = q['video_id'], float(q['start']), float(q['end'])
    events = [{'event': CHA_NAME.get(c, c), 'class': c, 'start': round(max(s, cs) - cs, 2),
               'end': round(min(e, ce) - cs, 2)}
              for c, s, e in CHA_ACTIONS.get(vid, []) if e > cs and s < ce]
    fps = FPS.get(vid + '.mp4') or 30.0
    frames = []
    for f in sorted(q['situations'], key=int):
        s = q['situations'][f]
        frames.append({'t': round(int(f) / fps - cs, 2),
                       'objects': [OBJ.get(o, o) for o in s.get('bbox_labels', [])],
                       'relations': [[OBJ.get(a, a), REL.get(l, l), OBJ.get(b, b)]
                                     for (a, b), l in zip(s.get('rel_pairs', []), s.get('rel_labels', []))],
                       'actions': [ACTION_NAME.get(a, a) for a in s.get('actions', [])]})
    return {'uid': r.uid, 'video_id': vid, 'duration_s': round(ce - cs, 2),
            'events': sorted(events, key=lambda e: e['start']), 'keyframes': frames}
NOISE_VERBS = {'appears', 'appear', 'disappears', 'disappear', 'is visible', 'visible', 'is seen', 'seen',
               'is present', 'present', 'exists', 'is shown', 'shown', 'comes into view', 'leaves view'}
WORN = {'glasses', 'sunglasses', 'hat', 'cap', 'helmet', 'shoe', 'shoes', 'sock', 'socks', 'shirt', 'jacket', 'coat',
        'dress', 'scarf', 'glove', 'gloves', 'ring', 'necklace', 'bracelet', 'watch', 'tie', 'mask', 'costume',
        'diaper', 'bib', 'pants', 'shorts', 'skirt', 'boot', 'boots'}
MAX_INTERVALS = 8

def state_phrase(state, target):
    return {'held_by': f'held by {target}', 'on': f'on {target}', 'near': f'near {target}',
            'absent': 'not visible', 'visible': 'visible'}.get(state, state)

def facts_from_cv(ev, cap):
    events, seen = [], set()
    for e in ev['events']:
        verb = (e.get('verb') or '').strip().lower()
        if not verb or verb in NOISE_VERBS:
            continue
        who = e.get('actor') or e.get('actor_text') or 'someone'
        obj = e.get('object') or e.get('object_text')
        loc = e.get('location')
        if loc and loc.strip().lower() in ('none', '-', ''):
            loc = None
        key = (round(e['t0'], 1), round(e['t1'], 1), who, verb, obj)
        if key in seen:
            continue
        seen.add(key)
        events.append({'t0': float(e['t0']), 't1': float(e['t1']), 'who': who, 'verb': verb, 'obj': obj, 'loc': loc})
    events.sort(key=lambda x: (x['t0'], x['t1']))
    used = {x['who'] for x in events} | {x['obj'] for x in events if x['obj']}
    confirmed = {c['obj'] for c in ev['changes'] if c.get('status') == 'confirmed'}
    objects, changes = {}, []
    for oid, o in ev['objects'].items():
        if o.get('scene') or o['cls'] in WORN or (oid not in used and oid not in confirmed):
            continue
        ivs = []
        for iv in o['intervals']:
            ivs.append({'t0': iv['t0'], 't1': iv['t1'], 'text': state_phrase(iv['state'], iv['target']),
                        'status': 'uncertain' if iv.get('status') == 'uncertain' else 'ok'})
        objects[oid] = {'cls': o['cls'], 'intervals': ivs}
        for prev, cur in zip(ivs, ivs[1:]):
            changes.append({'t': cur['t0'], 'obj': oid, 'from': prev['text'], 'to': cur['text'], 'status': cur['status']})
    changes.sort(key=lambda c: (c['t'], c['obj']))
    captions = [{'t0': c['ws'], 't1': c['we'], 'text': c['text']} for c in (cap or {}).get('captions', []) if c.get('text')]
    return {'duration': float(ev['duration_s']), 'events': events, 'objects': objects, 'changes': changes,
            'captions': captions, 'source': 'cv'}

def facts_from_oracle(orc):
    events = [{'t0': float(e['start']), 't1': float(e['end']), 'who': 'person',
               'verb': e['event'][0].lower() + e['event'][1:], 'obj': None, 'loc': None} for e in orc['events']]
    events.sort(key=lambda x: (x['t0'], x['t1']))
    # relationship triples -> intervals over consecutive keyframes
    runs = {}
    kfs = sorted(orc['keyframes'], key=lambda k: k['t'])
    for i, k in enumerate(kfs):
        for a, rel, b in k['relations']:
            key = (b, f'{a} {rel.replace("_", " ")} {b}')
            r = runs.setdefault(key, [])
            if r and r[-1][2] == i - 1:
                r[-1][1], r[-1][2] = k['t'], i
            else:
                r.append([k['t'], k['t'], i])
    objects, changes = {}, []
    for (b, text), rr in runs.items():
        o = objects.setdefault(b, {'cls': b, 'intervals': []})
        for t0, t1, _ in rr:
            o['intervals'].append({'t0': t0, 't1': t1, 'text': text, 'status': 'ok'})
    for b, o in objects.items():
        o['intervals'].sort(key=lambda x: (x['t0'], x['text']))
        for iv in o['intervals']:
            changes.append({'t': iv['t0'], 'obj': b, 'from': None, 'to': iv['text'], 'status': 'ok'})
    changes.sort(key=lambda c: (c['t'], c['obj']))
    return {'duration': float(orc['duration_s']), 'events': events, 'objects': objects, 'changes': changes,
            'captions': [], 'source': 'oracle'}
TOL = CFG['memory'].get('overlap_tolerance_s', 0.5)
COOC = CFG['memory'].get('cooccur_window_s', 1.0)

def words_for_ids(text):
    return re.sub(r'\b([a-z]+(?:_[a-z]+)*?)_(?:ov)?(\d+)\b', lambda m: m.group(1).replace('_', ' ') + ' ' + m.group(2), text)

def tfmt(a, b=None):
    return f'{a:.1f}s' if b is None else f'{a:.1f}-{b:.1f}s'

def ev_text(e):
    s = f"{e['who']} {e['verb']}" + (f" {e['obj']}" if e['obj'] else '')
    return s + (f" ({e['loc']})" if e['loc'] else '')

def items_events(F):
    return [{'id': f'E{i}', 'kind': 'event', 't0': e['t0'], 't1': e['t1'],
             'text': f"E{i} [t={tfmt(e['t0'], e['t1'])}] {ev_text(e)}"} for i, e in enumerate(F['events'], 1)]

def items_states(F):
    out = []
    for oid, o in sorted(F['objects'].items(), key=lambda kv: (kv[1]['intervals'][0]['t0'] if kv[1]['intervals'] else 0, kv[0])):
        ivs = o['intervals']
        if not ivs or all(iv['text'] in ('visible', 'not visible') for iv in ivs) and F['source'] == 'cv' and len(ivs) <= 1:
            continue
        parts = [f"{tfmt(iv['t0'], iv['t1'])} {iv['text']}" + (' (uncertain)' if iv['status'] == 'uncertain' else '')
                 for iv in ivs[:MAX_INTERVALS]]
        if len(ivs) > MAX_INTERVALS:
            parts.append('...')
        label = oid if F['source'] == 'oracle' or o['cls'].replace(' ', '_') in oid else f"{oid} ({o['cls']})"
        out.append({'id': f'S_{oid}', 'kind': 'state', 't0': ivs[0]['t0'], 't1': ivs[-1]['t1'],
                    'text': f'{label} | ' + ' | '.join(parts)})
    return out

def render_M1(F):
    sents = []
    for o_id, o in F['objects'].items():
        if o['intervals'] and F['source'] == 'cv':
            iv = o['intervals'][0]
            if iv['t0'] <= 0.5 and iv['text'] != 'not visible':
                sents.append((0.0, 0, f"At the start, {o_id} is {iv['text']}."))
    for e in F['events']:
        tail = f" {e['obj']}" if e['obj'] else ''
        place = f" at {e['loc']}" if e['loc'] else ''
        sents.append((e['t0'], 1, f"From {e['t0']:.1f} s to {e['t1']:.1f} s, {e['who']} {e['verb']}{tail}{place}."))
    for c in F['changes']:
        if F['source'] == 'oracle':
            sents.append((c['t'], 2, f"At {c['t']:.1f} s: {c['to']}."))
        else:
            verb = 'may now be' if c['status'] == 'uncertain' else 'is now'
            sents.append((c['t'], 2, f"At {c['t']:.1f} s, {c['obj']} {verb} {c['to']}."))
    sents.sort(key=lambda x: (x[0], x[1]))
    items = [{'id': f'P{i}', 'kind': 'sentence', 't0': t, 't1': t, 'text': words_for_ids(s)}
             for i, (t, _, s) in enumerate(sents, 1)]
    return items, [('', items)]

def render_M2(F):
    ev = items_events(F)
    return ev, [('EVENTS', ev)]

def render_M3(F):
    ev, st = items_events(F), items_states(F)
    return ev + st, [('EVENTS', ev), ('OBJECT STATES', st)]

def allen(a, b):
    if a[1] < b[0] - TOL:
        return 'BEFORE'
    if b[1] < a[0] - TOL:
        return 'AFTER'
    if a[0] >= b[0] - TOL and a[1] <= b[1] + TOL and (a[1] - a[0]) < (b[1] - b[0]) - TOL:
        return 'DURING'
    return 'OVERLAPS'

def summary_item(F):
    people = sorted({e['who'] for e in F['events'] if str(e['who']).startswith('person')})
    from collections import Counter
    verbs = Counter(e['verb'] for e in F['events']).most_common(4)
    objs = Counter(e['obj'] for e in F['events'] if e['obj']).most_common(4)
    s = (f"Video of {F['duration']:.0f} s with {len(F['events'])} events and {len(F['changes'])} state changes. "
         f"People: {', '.join(people) or 'none detected'}. "
         f"Main objects: {', '.join(o for o, _ in objs) or 'none'}. "
         f"Most frequent actions: {', '.join(f'{v} ({n})' for v, n in verbs) or 'none'}.")
    return {'id': 'SUM', 'kind': 'summary', 't0': 0.0, 't1': F['duration'], 'text': s}

def render_M4(F):
    ev, st = items_events(F), items_states(F)
    segs, seg_len = [], 10.0
    n_seg = max(1, int(np.ceil(F['duration'] / seg_len)))
    for k in range(n_seg):
        a, b = k * seg_len, min((k + 1) * seg_len, F['duration'])
        ids = [x['id'] for x in ev if a <= x['t0'] < b]
        who = sorted({e['who'] for e, x in zip(F['events'], ev) if x['id'] in ids})
        segs.append({'id': f'SEG{k + 1}', 'kind': 'segment', 't0': a, 't1': b,
                     'text': f"SEG{k + 1} [t={tfmt(a, b)}] events {', '.join(ids) if ids else 'none'}"
                             + (f"; who: {', '.join(who)}" if who else '')})
    rel = []
    E = F['events']
    for i in range(len(E) - 1):
        a, b = (E[i]['t0'], E[i]['t1']), (E[i + 1]['t0'], E[i + 1]['t1'])
        rel.append({'id': f'R{len(rel) + 1}', 'kind': 'relation', 't0': min(a[0], b[0]), 't1': max(a[1], b[1]),
                    'text': f'E{i + 1} {allen(a, b)} E{i + 2}'})
    for i, x in enumerate(E):
        for j, y in enumerate(E):
            if i != j and j != i + 1 and i != j + 1 and allen((x['t0'], x['t1']), (y['t0'], y['t1'])) == 'DURING':
                rel.append({'id': f'R{len(rel) + 1}', 'kind': 'relation', 't0': y['t0'], 't1': y['t1'],
                            'text': f'E{i + 1} DURING E{j + 1}'})
    for c in F['changes']:
        for i, e in enumerate(E, 1):
            if c['obj'] in (e['who'], e['obj']) and abs(c['t'] - e['t0']) <= COOC:
                what = f"{c['obj']} becomes {c['to']}" if F['source'] == 'cv' else c['to']
                rel.append({'id': f'R{len(rel) + 1}', 'kind': 'relation', 't0': c['t'], 't1': e['t1'],
                            'text': f"[t={tfmt(c['t'])}] {what} CO_OCCURS E{i}"})
    summ = summary_item(F)
    return [summ] + segs + ev + st + rel, [('SUMMARY', [summ]), ('SEGMENTS', segs), ('EVENTS', ev),
                                           ('OBJECT STATES', st), ('RELATIONS', rel)]

def render_CAP(F):
    it = [{'id': f'C{i}', 'kind': 'caption', 't0': c['t0'], 't1': c['t1'], 'text': f"[t={tfmt(c['t0'], c['t1'])}] {c['text']}"}
          for i, c in enumerate(F['captions'], 1)]
    return it, [('CAPTIONS', it)]

def render_VA(F):
    caps, _ = render_CAP(F)
    table = []
    for oid, o in sorted(F['objects'].items()):
        seen = [iv for iv in o['intervals'] if iv['text'] != 'not visible']
        if not seen:
            continue
        states = sorted({iv['text'] for iv in seen if iv['text'] != 'visible'})
        table.append({'id': f'O_{oid}', 'kind': 'object', 't0': seen[0]['t0'], 't1': seen[-1]['t1'],
                      'text': f"{oid} | {o['cls']} | seen {tfmt(seen[0]['t0'], seen[-1]['t1'])} | "
                              f"states: {', '.join(states) if states else 'none'}"})
    for p in sorted({e['who'] for e in F['events'] if str(e['who']).startswith('person')}):
        ts = [(e['t0'], e['t1']) for e in F['events'] if e['who'] == p]
        table.append({'id': f'O_{p}', 'kind': 'object', 't0': min(a for a, _ in ts), 't1': max(b for _, b in ts),
                      'text': f"{p} | person | active {tfmt(min(a for a, _ in ts), max(b for _, b in ts))}"})
    return caps + table, [('SEGMENT CAPTIONS', caps), ('OBJECT TABLE', table)]

RENDER = {'M1': render_M1, 'M2': render_M2, 'M3': render_M3, 'M4': render_M4, 'CAP': render_CAP, 'VA': render_VA}

def to_text(sections):
    blocks = []
    for head, items in sections:
        lines = [x['text'] for x in items]
        if not lines:
            continue
        blocks.append((f'{head}:\n' if head else '') + ('\n'.join(lines) if head else ' '.join(lines)))
    return '\n\n'.join(blocks) if blocks else '(nothing detected)'


def stems_of(text):
    return {w[:-1] if w.endswith('s') and len(w) > 3 else w for w in re.findall(r'[a-z]+', text.lower())} - \
        {'the', 'a', 'an', 'some', 'something', 'person', 'to', 'of', 'on', 'in', 'at', 'with', 'from', 'and', 'it',
         'e', 't', 's', 'seg', 'events', 'none'}

def evidence_for(items, answer_text):
    """Memory items sharing the most words with the correct option (at most 2), or none."""
    aw = stems_of(answer_text)
    scored = sorted(((len(aw & stems_of(it['text'])), i, it['id']) for i, it in enumerate(items)
                     if it['kind'] in ('event', 'state', 'relation', 'sentence')), key=lambda x: (-x[0], x[1]))
    if not scored or scored[0][0] < min(2, len(aw)):
        return []
    return [iid for s, _, iid in scored[:2] if s == scored[0][0]]

def star_train_candidates():
    """Stream STAR_train.json and keep a fixed-seed sample of usable questions."""
    path = None
    for p in [P['source'] / 'star' / 'STAR_train.json', P['local_tmp'] / 'STAR_train.json']:
        if p.exists() and p.stat().st_size > 0:
            path = p; break
    if path is None:
        path = P['local_tmp'] / 'STAR_train.json'
        log(f'downloading STAR_train.json from {STAR_TRAIN_URL}')
        subprocess.run(['wget', '-q', '-c', '-O', str(path), STAR_TRAIN_URL], check=True, timeout=3 * 3600)
        log(f'STAR_train.json: {_gb(path.stat().st_size)} GB')
    pip_install('ijson')
    import ijson
    test_videos = set()
    for name in ['star_main', 'star_gap']:
        f = P['subsets'] / f'{name}.csv'
        if f.exists():
            test_videos |= set(pd.read_csv(f, dtype=str).video_id)
    rng = random.Random(CFG['seed'] + 7)
    keep, seen = [], 0
    with open(path, 'rb') as f:
        for q in ijson.items(f, 'item', use_float=True):
            qtype = q['question_id'].split('_')[0]
            if qtype not in ('Sequence', 'Interaction') or q['video_id'] in test_videos:
                continue
            opts = [c['choice'] for c in q.get('choices', [])]
            if q.get('answer') not in opts or not q.get('situations'):
                continue
            seen += 1                                            # reservoir sample of 3x the target size
            if len(keep) < 3 * N_STAR_TRAIN:
                keep.append(q)
            else:
                j = rng.randrange(seen)
                if j < len(keep):
                    keep[j] = q
    rng.shuffle(keep)
    out, per_video = [], {}
    for q in keep:
        if per_video.get(q['video_id'], 0) >= 2:
            continue
        per_video[q['video_id']] = per_video.get(q['video_id'], 0) + 1
        out.append(q)
        if len(out) >= N_STAR_TRAIN:
            break
    log(f'STAR train: {seen} usable questions (test videos excluded), {len(out)} sampled')
    return out

def build_train_set():
    rows = []
    for q in star_train_candidates():
        opts = [c['choice'].rstrip('.') for c in q['choices']]
        gold = [c['choice'] for c in q['choices']].index(q['answer'])
        uid = 'star_train_' + q['question_id']
        orc = oracle_record(q, types.SimpleNamespace(uid=uid))
        if not orc['events'] and not orc['keyframes']:
            continue
        items, sections = render_M4(facts_from_oracle(orc))
        rows.append({'uid': uid, 'source': 'star_train_oracle', 'question': q['question'], 'options': opts,
                     'gold': gold, 'memory': to_text(sections), 'evidence': evidence_for(items, opts[gold])})
    if USE_DEV:
        n_dev = 0
        for q in QSETS['dev']:
            f = MEM_LOCAL / 'M4' / f"{q['key']}.json"
            if not f.exists():
                continue
            m = memory('M4', q['key'])
            rows.append({'uid': q['uid'], 'source': 'nextqa_dev_cv', 'question': q['question'], 'options': q['options'],
                         'gold': q['gold'], 'memory': m['text'], 'evidence': evidence_for(m['items'], q['options'][q['gold']])})
            n_dev += 1
        log(f'NExT-QA dev: {n_dev} questions with an M4 memory')
    random.Random(CFG['seed'] + 8).shuffle(rows)
    return rows

if TRAIN_SET.exists():
    TRAIN_ROWS = read_jsonl(TRAIN_SET)
    print(f'training set from Drive: {len(TRAIN_ROWS)} examples')
else:
    TRAIN_ROWS = build_train_set()
    tmp = QL_LOCAL / TRAIN_SET.name
    tmp.unlink(missing_ok=True)
    for r in TRAIN_ROWS:
        append_jsonl(tmp, r)
    put_on_drive(tmp, TRAIN_SET.name)
    log(f'training set saved: {len(TRAIN_ROWS)} examples')
TRAIN_HASH = hashlib.sha1(json.dumps([r['uid'] for r in TRAIN_ROWS]).encode()).hexdigest()[:12]
print(pd.Series([r['source'] for r in TRAIN_ROWS]).value_counts().to_string())
print('with evidence IDs:', round(np.mean([bool(r['evidence']) for r in TRAIN_ROWS]), 2), '| set hash', TRAIN_HASH)
print('\nexample memory (first 600 characters):\n', TRAIN_ROWS[0]['memory'][:600])

STAR actions mapped to Charades classes: 111/111 (111 exact, 0 fuzzy)
training set from Drive: 900 examples
star_train_oracle    600
nextqa_dev_cv        300
with evidence IDs: 0.29 | set hash c482d65e0beb

example memory (first 600 characters):
 SUMMARY:
Video of 5 s with 7 events and 20 state changes. People: person. Main objects: none. Most frequent actions: taking food from somewhere (1), taking a dish/es from somewhere (1), closing a door (1), holding some food (1).

SEGMENTS:
SEG1 [t=0.0-5.4s] events E1, E2, E3, E4, E5, E6, E7; who: person

EVENTS:
E1 [t=0.0-2.4s] person taking food from somewhere
E2 [t=0.0-2.6s] person taking a dish/es from somewhere
E3 [t=0.0-5.2s] person closing a door
E4 [t=0.0-5.4s] person holding some food
E5 [t=0.0-5.4s] person holding a dish
E6 [t=0.0-5.4s] person closing a refrigerator
E7 [t=3.7-5.4s] pe


## 4. Models, prompts and strategy A

The prompts and strategy A are 05's own code, so the tuned model is asked exactly what the untuned models are asked.

In [6]:
if not MOCK:
    pip_install('transformers>=4.57.0', 'accelerate>=1.0', 'bitsandbytes>=0.45', 'peft>=0.13')
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()

def load_llm(spec):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    kw = {'device_map': {'': 0}, 'low_cpu_mem_usage': True, 'revision': spec.get('revision', 'main')}
    if spec.get('quant') == 'nf4':
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    else:
        kw['dtype'] = torch.float16
    tok = AutoTokenizer.from_pretrained(spec['id'], revision=spec.get('revision', 'main'))
    try:
        model = AutoModelForCausalLM.from_pretrained(spec['id'], **kw)
    except (ValueError, KeyError):                       # Gemma 3 4B is multimodal
        from transformers import AutoModelForImageTextToText
        model = AutoModelForImageTextToText.from_pretrained(spec['id'], **kw)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    devs = {str(p.device) for p in model.parameters()}
    if any(not d.startswith('cuda') for d in devs):
        raise RuntimeError(f'model not fully on the GPU ({devs}): Disconnect and delete runtime, then Run all once.')
    return model.eval(), tok

_GEN_CAP = {}   # learned batch size per (model, task length): survives between calls, so it does not run out of memory again and again
def llm_generate(model, tok, convs, max_new_tokens, temperature=0.0, batch_size=8, max_batch_tokens=7000):
    """Batched chat generation (thinking off), token-budgeted, with OOM backoff. Returns (texts, prompt_lens, gen_lens)."""
    import torch, gc
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False) for m in convs]
    lens = [len(tok(t)['input_ids']) for t in texts]
    order = sorted(range(len(texts)), key=lambda k: lens[k])
    outs, glens = [None] * len(texts), [0] * len(texts)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
    gen.update(dict(do_sample=True, temperature=temperature, top_p=0.95) if temperature > 0 else dict(do_sample=False))
    key = (id(model), max_new_tokens)
    cap = min(batch_size, _GEN_CAP.get(key, batch_size))
    i, ok_streak = 0, 0
    while i < len(order):
        j = i + 1
        while j < len(order) and j - i < cap and (j - i + 1) * (lens[order[j]] + max_new_tokens) <= max_batch_tokens:
            j += 1
        idx = order[i:j]
        oom = False
        try:
            enc = tok([texts[k] for k in idx], return_tensors='pt', padding=True).to(model.device)
            with torch.inference_mode():
                out = model.generate(**enc, **gen)
            new = out[:, enc['input_ids'].shape[1]:]
            for k, row, d in zip(idx, new, tok.batch_decode(new, skip_special_tokens=True)):
                outs[k], glens[k] = d, int((row != tok.pad_token_id).sum())
            del enc, out, new
            i = j
            if len(idx) > _GEN_CAP.get(key, batch_size):    # a bigger batch worked: remember it
                _GEN_CAP[key] = len(idx)
            ok_streak += 1
            if ok_streak >= 4 and cap < batch_size:        # try one bigger again after 4 good batches
                cap += 1; ok_streak = 0
        except torch.cuda.OutOfMemoryError:
            oom = True
        if oom:                                           # free memory OUTSIDE the except block, or it stays held
            enc = out = new = None
            gc.collect(); torch.cuda.empty_cache()
            if len(idx) == 1:
                raise torch.cuda.OutOfMemoryError('one prompt alone does not fit on the GPU')
            cap = max(1, len(idx) // 2); ok_streak = 0
            _GEN_CAP[key] = cap
            log(f'GPU memory full at batch {len(idx)}: retrying with batch {cap}', 'WARN')
    return outs, lens, glens

if MOCK:
    class _T:
        pad_token_id = 0
        def __call__(self, t):
            return {'input_ids': t.split()}
        def apply_chat_template(self, m, **k):
            return ' '.join(x['content'] for x in m)
    def load_llm(spec):
        return None, _T()
    def llm_generate(model, tok, convs, max_new_tokens, temperature=0.0, batch_size=8, max_batch_tokens=7000):
        outs = []
        for m in convs:
            u = m[-1]['content']
            ids = re.findall(r'\b(E\d+|P\d+|C\d+)\b', m[-2]['content'] if len(m) > 2 else u) or ['E1']
            if 'supported' in u:
                outs.append('{"supported": %s}' % ('true' if len(u) % 3 else 'false'))
            elif 'sub_questions' in u:
                outs.append('{"type": "temporal", "target_object": "cup", "target_actor": "person", "anchor_event": "pick up", '
                            '"relation": "after", "sub_questions": ["What happens first?", "What happens next?"]}')
            else:
                letters = re.findall(r'^([A-E])\. ', u, flags=re.M) or ['A']
                outs.append('{"answer": "%s", "evidence": ["%s"], "reasoning_steps": ["checked"]}' % (letters[len(u) % len(letters)], ids[0]))
        return outs, [len(' '.join(x['content'] for x in m).split()) for m in convs], [20] * len(convs)

EMB = None                                   # strategy A uses no retrieval
LET = 'ABCDE'
SYS = ('You answer multiple-choice questions about a video using only the video memory provided. The memory was '
       'produced automatically by computer vision and may contain errors or omissions. Reply with JSON only.')
BUDGET = CFG['llm']['retrieval_token_budget']
EXPAND = CFG['llm']['expansion_max_items']

def options_text(q):
    return '\n'.join(f'{LET[i]}. {o}' for i, o in enumerate(q['options']))

def parse_json(text):
    """The first JSON object (dict) in a reply, or None. Lists and other JSON values are ignored."""
    if not isinstance(text, str):
        return None
    text = re.sub(r'<think>.*?</think>', '', text, flags=re.S)
    s = text.find('{')
    while s != -1:
        depth = 0
        for j in range(s, len(text)):
            depth += (text[j] == '{') - (text[j] == '}')
            if depth == 0:
                try:
                    v = json.loads(text[s:j + 1])
                    if isinstance(v, dict):
                        return v
                except json.JSONDecodeError:
                    pass
                break
        s = text.find('{', s + 1)
    return None

def letter_of(js, q):
    if not isinstance(js, dict):
        return None
    a = js.get('answer', '')
    if isinstance(a, list):
        a = a[0] if a else ''
    a = str(a).strip().upper()[:1]
    return a if a and a in LET[:len(q['options'])] else None

ANSWER_FMT = 'Reply as {"answer": "<letter>", "evidence": ["<IDs of the memory items you used, e.g. E3>"]}.'
ANSWER_FMT_STEPS = ('Reply as {"answer": "<letter>", "evidence": ["<IDs of the memory items you used>"], '
                    '"reasoning_steps": ["<at most 3 short steps>"]}.')

def mem_text_capped(m, cap_tokens=3000):
    """The full memory for strategy A/B, cut at about cap_tokens (pilot rule keeps prompts inside a T4)."""
    if m['tokens'] <= cap_tokens:
        return m['text'], False
    words = m['text'].split(' ')
    keep = int(len(words) * cap_tokens / m['tokens'])
    return ' '.join(words[:keep]) + '\n(... memory cut to fit)', True

def conv_answer(memtext, q, header='VIDEO MEMORY', extra='', steps=False):
    return [{'role': 'system', 'content': SYS},
            {'role': 'user', 'content': f'{header}:\n{memtext}\n\n{extra}QUESTION: {q["question"]}\nOPTIONS:\n{options_text(q)}\n\n'
                                        + (ANSWER_FMT_STEPS if steps else ANSWER_FMT)}]

def conv_parse(q):
    return [{'role': 'user', 'content':
             f'Analyse this question about a video. QUESTION: {q["question"]}\nOPTIONS:\n{options_text(q)}\n\n'
             'Reply with JSON only: {"type": "causal|temporal_before|temporal_after|temporal_during|descriptive", '
             '"target_object": "...", "target_actor": "...", "anchor_event": "...", "relation": "before|after|during|why|how|none", '
             '"sub_questions": ["2 to 4 short questions that, answered in order, answer the main question"]}'}]

def conv_verify(items_text, q, letter, evidence):
    opt = q['options'][LET.index(letter)] if letter else '(none)'
    return [{'role': 'user', 'content':
             f'MEMORY ITEMS:\n{items_text}\n\nQUESTION: {q["question"]}\nPROPOSED ANSWER: {opt}\n'
             f'CITED ITEMS: {", ".join(evidence) or "none"}\n\nDo the memory items support the proposed answer? '
             'Reply with JSON only: {"supported": true or false}'}]

# ---- retrieval ---------------------------------------------------------------------------------
_ITEM_TOK, _EMB_CACHE = {}, {}
def item_tokens(text):
    if text not in _ITEM_TOK:
        _ITEM_TOK[text] = len(TOK(text)['input_ids'])
    return _ITEM_TOK[text]

def embed(texts):
    import torch
    tk, md_ = EMB
    out = []
    for i in range(0, len(texts), 64):
        enc = tk(texts[i:i + 64], padding=True, truncation=True, max_length=256, return_tensors='pt').to('cuda')
        with torch.inference_mode():
            v = md_(**enc).last_hidden_state[:, 0]
        out.append(torch.nn.functional.normalize(v, dim=-1).float().cpu().numpy())
    return np.concatenate(out)

def retrieve(m, query, budget):
    items = m['items']
    if not items:
        return [], 0
    toks = [re.findall(r'[a-z0-9]+', it['text'].lower()) for it in items]
    qt = re.findall(r'[a-z0-9]+', query.lower())
    try:
        from rank_bm25 import BM25Okapi
        bm = np.array(BM25Okapi(toks).get_scores(qt))
    except Exception:
        bm = np.array([len(set(t) & set(qt)) for t in toks], float)
    score = (bm - bm.min()) / (bm.max() - bm.min() + 1e-9)
    if EMB is not None:
        ck = (m['format'], m['key'])
        if ck not in _EMB_CACHE:
            _EMB_CACHE[ck] = embed([it['text'] for it in items])
        sim = _EMB_CACHE[ck] @ embed([query])[0]
        score = 0.5 * score + 0.5 * (sim - sim.min()) / (sim.max() - sim.min() + 1e-9)
    order = list(np.argsort(-score))
    chosen, used = [], 0
    for k in order:
        t = item_tokens(items[k]['text'])
        if used + t > budget:
            continue
        chosen.append(k); used += t
    ev_idx = [k for k in order[:2] if items[k]['kind'] == 'event']
    if len(ev_idx) == 2:                                   # temporal expansion between the two best events
        a, b = sorted(ev_idx)
        between = [k for k in range(a + 1, b) if items[k]['kind'] == 'event' and k not in chosen]
        if 0 < len(between) <= EXPAND:
            extra = sum(item_tokens(items[k]['text']) for k in between)
            if used + extra <= budget * 1.25:
                chosen += between; used += extra
    chosen.sort(key=lambda k: (items[k]['t0'], k))
    return [items[k] for k in chosen], used

def query_of(q, parse):
    p = parse or {}
    parts = [q['question'], p.get('anchor_event') or '', p.get('target_object') or '', p.get('target_actor') or '',
             ' '.join(p.get('sub_questions') or []), ' '.join(q['options'])]
    return ' '.join(str(x) for x in parts if x)

# ---- strategies (all batched over a chunk of questions) ----------------------------------------
GEN_BATCH, GEN_BATCH_TOKENS = 8, 8000   # fits a T4; the learned batch size is kept between calls
def gen(convs, max_new, model=None, tok=None, temperature=0.0):
    return llm_generate(model or LLM, tok or TOK, convs, max_new, temperature,
                        batch_size=GEN_BATCH, max_batch_tokens=GEN_BATCH_TOKENS)

def answer_with_retry(convs, qs, recs, max_new, model=None, tok=None):
    outs, pl, gl = gen(convs, max_new, model, tok)
    for r, p, g in zip(recs, pl, gl):
        r['calls'] += 1; r['prompt_tokens'] += p; r['gen_tokens'] += g
    res = [(letter_of(parse_json(o), q), parse_json(o), o) for o, q in zip(outs, qs)]
    bad = [i for i, (L, js, o) in enumerate(res) if L is None]
    if bad:
        retry = [convs[i] + [{'role': 'assistant', 'content': res[i][2] or ''},
                             {'role': 'user', 'content': 'That was not valid. ' + ANSWER_FMT + ' JSON only.'}] for i in bad]
        outs2, pl2, gl2 = gen(retry, max_new, model, tok)
        for i, o, p, g in zip(bad, outs2, pl2, gl2):
            recs[i]['calls'] += 1; recs[i]['prompt_tokens'] += p; recs[i]['gen_tokens'] += g; recs[i]['retried'] = True
            js = parse_json(o)
            res[i] = (letter_of(js, qs[i]), js, o)
    return res

def do_parse(qs, recs):
    outs, pl, gl = gen([conv_parse(q) for q in qs], 128)
    parses = []
    for r, o, p, g in zip(recs, outs, pl, gl):
        r['calls'] += 1; r['prompt_tokens'] += p; r['gen_tokens'] += g
        js = parse_json(o)
        parses.append(js if isinstance(js, dict) else {})
        r['parse'] = parses[-1]
    return parses

def finish(r, q, L, js, raw):
    r.update(pred=L, gold=LET[q['gold']], correct=bool(L == LET[q['gold']]), valid_json=L is not None,
             evidence=[str(x) for x in ((js or {}).get('evidence') or [])][:10] if isinstance(js, dict) else [],
             steps=(js or {}).get('reasoning_steps') if isinstance(js, dict) else None,
             raw=None if L is not None else (raw or '')[:300])

def run_chunk(run, qs):
    strat = run['strategy']
    mems = [memory(run['format'], q['key']) for q in qs]
    recs = [{'uid': q['uid'], 'run': run['id'], 'qset': run['qset'], 'format': run['format'], 'strategy': strat,
             'key': q['key'], 'type': q['type'], 'model': MAIN_SPEC['id'], 'calls': 0, 'prompt_tokens': 0,
             'gen_tokens': 0, 'mem_tokens': m['tokens']} for q, m in zip(qs, mems)]
    if strat in ('A', 'A-SC'):
        texts = [mem_text_capped(m) for m in mems]
        for r, (_, cut) in zip(recs, texts):
            r['memory_cut'] = cut
        convs = [conv_answer(t, q) for (t, _), q in zip(texts, qs)]
        if strat == 'A':
            for r, q, (L, js, raw) in zip(recs, qs, answer_with_retry(convs, qs, recs, 64)):
                finish(r, q, L, js, raw)
        else:
            n = CFG['llm']['self_consistency_samples']
            outs, pl, gl = gen([c for c in convs for _ in range(n)], 64,
                               temperature=CFG['llm']['self_consistency_temperature'])
            for i, (r, q) in enumerate(zip(recs, qs)):
                votes = [letter_of(parse_json(o), q) for o in outs[i * n:(i + 1) * n]]
                r['calls'] += n; r['prompt_tokens'] += sum(pl[i * n:(i + 1) * n]); r['gen_tokens'] += sum(gl[i * n:(i + 1) * n])
                valid = [v for v in votes if v]
                L = max(valid, key=valid.count) if valid else None     # ties: earliest sample
                finish(r, q, L, parse_json(outs[i * n]), outs[i * n])
                r['votes'] = votes
        return recs
    parses = do_parse(qs, recs)
    if strat == 'B':
        texts = [mem_text_capped(m) for m in mems]
        convs = [conv_answer(t, q, extra='SUB-QUESTIONS (answer them in order, then the main question):\n'
                             + '\n'.join(f'- {s}' for s in (p.get('sub_questions') or [])[:4]) + '\n\n', steps=True)
                 for (t, _), q, p in zip(texts, qs, parses)]
        for r, q, (L, js, raw) in zip(recs, qs, answer_with_retry(convs, qs, recs, 160)):
            finish(r, q, L, js, raw)
        return recs
    # C, D, D-X: retrieval
    shown = []
    for r, m, q, p in zip(recs, mems, qs, parses):
        items, used = retrieve(m, query_of(q, p), BUDGET)
        shown.append(items)
        r['retrieved'] = [it['id'] for it in items]; r['retrieved_tokens'] = used
    convs = [conv_answer('\n'.join(it['text'] for it in its) or '(no items)', q, header='RETRIEVED MEMORY ITEMS', steps=True)
             for its, q in zip(shown, qs)]
    res = answer_with_retry(convs, qs, recs, 128)
    if strat == 'C':
        for r, q, (L, js, raw) in zip(recs, qs, res):
            finish(r, q, L, js, raw)
        return recs
    # verification
    vm, vt = (VERIFIER if strat == 'D-X' else (LLM, TOK))
    vconvs = [conv_verify('\n'.join(it['text'] for it in its), q, L, [str(e) for e in ((js or {}).get('evidence') or [])]
                          if isinstance(js, dict) else [])
              for its, q, (L, js, raw) in zip(shown, qs, res)]
    vout, vpl, vgl = gen(vconvs, 24, vm, vt)
    redo = []
    for i, (r, its, (L, js, raw), o, p, g) in enumerate(zip(recs, shown, res, vout, vpl, vgl)):
        r['calls'] += 1; r['prompt_tokens'] += p; r['gen_tokens'] += g
        vj = parse_json(o)
        supported = bool(isinstance(vj, dict) and vj.get('supported') is True)
        cited = [str(e) for e in ((js or {}).get('evidence') or [])] if isinstance(js, dict) else []
        ids_ok = bool(cited) and all(c in {it['id'] for it in its} for c in cited)
        r.update(verifier_supported=supported, symbolic_ok=ids_ok, verifier_model=(VERIFIER_ID if strat == 'D-X' else MAIN_SPEC['id']))
        if not (supported and ids_ok and L):
            redo.append(i)
    final = list(res)
    if redo:
        items2 = []
        for i in redo:
            its, used = retrieve(mems[i], query_of(qs[i], parses[i]), BUDGET * 2)
            items2.append(its); recs[i]['retrieved_2'] = [it['id'] for it in its]
        convs2 = [conv_answer('\n'.join(it['text'] for it in its) or '(no items)', qs[i], header='RETRIEVED MEMORY ITEMS', steps=True)
                  for its, i in zip(items2, redo)]
        sub_recs = [recs[i] for i in redo]
        for i, out in zip(redo, answer_with_retry(convs2, [qs[i] for i in redo], sub_recs, 128)):
            recs[i]['reanswered'] = True
            final[i] = out if out[0] else final[i]       # keep the first answer if the second is invalid
    for r, q, (L, js, raw) in zip(recs, qs, final):
        finish(r, q, L, js, raw)
    return recs


SMALL_SPEC = CFG['models']['reasoner_small']     # Qwen3-4B
BIG_SPEC = CFG['models']['reasoner_main']        # Qwen3-8B
ADAPTER_FINAL = QL / '07_adapter_final.zip'
CKPT = QL / '07_checkpoint.zip'

def R(qset, fmt, strat, tag, spec):
    rid = f'{qset}.{fmt}.{strat}' + (f'.{tag}' if tag else '')
    return {'id': rid, 'qset': qset, 'format': fmt, 'strategy': strat, 'model_id': spec['id'] + ('+qlora' if tag == 'q4b-lora' else '')}
RUNS_BASE = [R(s, 'M4', 'A', 'q4b', SMALL_SPEC) for s in ['main', 'star']]
RUNS_LORA = [R(s, 'M4', 'A', 'q4b-lora', SMALL_SPEC) for s in ['main', 'star']]
RUNS_8B = [R('star', 'M4', 'A', '', BIG_SPEC)] if DO_8B_REFERENCE else []
ALL_RUNS = RUNS_BASE + RUNS_LORA + RUNS_8B

def todo_of(run):
    done = answered(run['id'])
    return [q for q in EVAL_Q[run['qset']] if q['uid'] not in done
            and (MEM_LOCAL / run['format'] / f"{q['key']}.json").exists()]

T_START, LAST_SYNC = time.time(), [time.time()]
BUDGET_S = CFG['runtime']['chunk_minutes'] * 60
def out_of_time():
    return time.time() - T_START > BUDGET_S

def answer_runs(runs):
    """Strategy A for these runs with the model now in LLM/TOK. Returns False if the time budget ran out."""
    global MAIN_SPEC
    for run in runs:
        todo = todo_of(run)
        if not todo:
            continue
        MAIN_SPEC = {'id': run['model_id']}       # recorded in each answer
        log(f'{run["id"]}: {len(todo)} questions to do')
        t0 = time.time()
        for c in range(0, len(todo), 48):
            if out_of_time():
                return False
            recs = run_chunk(run, todo[c:c + 48])
            per = (time.time() - t0) / max(1, c + len(recs))
            out = LOCAL_PRED / run['id'] / f's{SHARD}_{dt.datetime.now():%Y%m%d-%H%M%S-%f}.jsonl'
            out.parent.mkdir(parents=True, exist_ok=True)
            for r in recs:
                r['seconds'] = round(per, 2)
                append_jsonl(out, r)
            if time.time() - LAST_SYNC[0] > SYNC_EVERY_MIN * 60:
                sync_preds(); LAST_SYNC[0] = time.time()
        recs = [r for f in (LOCAL_PRED / run['id']).glob('*.jsonl') for r in read_jsonl(f)]
        log(f'{run["id"]}: done | {(time.time() - t0) / len(todo):.1f} s per question | accuracy {np.mean([r["correct"] for r in recs]):.3f}')
    return True

[14:33:34] INFO  pip install transformers>=4.57.0 accelerate>=1.0 bitsandbytes>=0.45 peft>=0.13


## 5. Training code

Each example is the strategy-A prompt (system + memory + question + options) followed by the target reply; the loss is computed on the reply only. The memory is shortened (word by word, from the end) until the example fits in `MAX_LEN` tokens.

The loop is written out (no `Trainer`) so it can stop at the session's time budget and resume exactly: the adapter, optimizer, gradient scaler, learning-rate schedule and position in the data are saved together as `outputs/qlora/07_checkpoint.zip`, read back from Drive to verify.

In [7]:
import torch, math

def target_text(row):
    return json.dumps({'answer': LET[row['gold']], 'evidence': row['evidence']})

def encode(row, tok):
    q = {'question': row['question'], 'options': row['options']}
    end = '<|im_end|>' if '<|im_end|>' in tok.get_vocab() else (tok.eos_token or '')
    tgt = tok(target_text(row) + end, add_special_tokens=False)['input_ids']
    words = row['memory'].split(' ')
    for _ in range(6):
        prompt = tok.apply_chat_template(conv_answer(' '.join(words), q), tokenize=False, add_generation_prompt=True,
                                         enable_thinking=False)
        p = tok(prompt, add_special_tokens=False)['input_ids']
        if len(p) + len(tgt) <= MAX_LEN:
            return p + tgt, [-100] * len(p) + tgt
        keep = int(len(words) * (MAX_LEN - len(tgt) - 64) / max(1, len(p))) - 5
        words = words[:max(10, keep)] + ['(...memory cut to fit)']
    return None, None

def lora_wrap(model):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    cfg = LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
                     target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'])
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model

def save_checkpoint(model, opt, scaler, sched, state, final=False):
    d = QL_LOCAL / ('final' if final else 'ckpt')
    shutil.rmtree(d, ignore_errors=True)
    model.save_pretrained(str(d))
    if not final:
        torch.save({'opt': opt.state_dict(), 'scaler': scaler.state_dict() if scaler else None,
                    'sched': sched.state_dict(), 'state': state}, d / 'train_state.pt')
    save_json({**state, 'train_hash': TRAIN_HASH, 'saved': dt.datetime.now().isoformat(timespec='seconds')}, d / 'progress.json')
    z = QL_LOCAL / (ADAPTER_FINAL.name if final else CKPT.name)
    with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in d.rglob('*'):
            if f.is_file():
                zf.write(f, f.relative_to(d))
    put_on_drive(z, z.name)
    log(f'{"final adapter" if final else "checkpoint"} saved to Drive: step {state["step"]}/{state["total_steps"]}')

def unzip_to(zp, d):
    shutil.rmtree(d, ignore_errors=True); d.mkdir(parents=True)
    local = QL_LOCAL / ('_' + zp.name)
    shutil.copyfile(zp, local)
    with zipfile.ZipFile(local) as zf:
        zf.extractall(d)
    return d

def train(model, tok):
    """QLoRA training with resume. Returns (peft model, finished?)."""
    from peft import set_peft_model_state_dict
    data = []
    for r in TRAIN_ROWS:
        ids, labels = encode(r, tok)
        if ids is not None:
            data.append((ids, labels))
    log(f'training examples encoded: {len(data)} | mean length {np.mean([len(x) for x, _ in data]):.0f} tokens')
    model = lora_wrap(model)
    dev = next(model.parameters()).device
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
    total_steps = math.ceil(len(data) * EPOCHS / GRAD_ACCUM)
    warm = max(1, int(0.05 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(           # linear warm-up (5%), then linear decay to 0
        opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total_steps - s) / max(1, total_steps - warm)))
    use_amp = dev.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda') if use_amp else None
    state = {'step': 0, 'micro': 0, 'total_steps': total_steps, 'losses': []}
    if CKPT.exists():
        d = unzip_to(CKPT, QL_LOCAL / 'resume')
        prog = load_json(d / 'progress.json', default={})
        if prog.get('train_hash') == TRAIN_HASH:
            from safetensors.torch import load_file
            set_peft_model_state_dict(model, load_file(str(d / 'adapter_model.safetensors')))
            ts = torch.load(d / 'train_state.pt', map_location='cpu', weights_only=False)
            opt.load_state_dict(ts['opt']); sched.load_state_dict(ts['sched'])
            if scaler and ts['scaler']:
                scaler.load_state_dict(ts['scaler'])
            state = ts['state']
            log(f'resumed training from Drive checkpoint: step {state["step"]}/{total_steps}')
        else:
            log('checkpoint on Drive belongs to a different training set: starting again', 'WARN')
    order = []
    for ep in range(EPOCHS):
        o = list(range(len(data))); random.Random(CFG['seed'] + 100 + ep).shuffle(o); order += o
    model.train()
    last_save, t0, run_loss = time.time(), time.time(), []
    micro = state['micro']
    while micro < len(order):
        ids, labels = data[order[micro]]
        x = torch.tensor([ids], device=dev); y = torch.tensor([labels], device=dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            loss = model(input_ids=x, labels=y).loss / GRAD_ACCUM
        if torch.isfinite(loss):
            (scaler.scale(loss) if scaler else loss).backward()
            run_loss.append(loss.item() * GRAD_ACCUM)
        micro += 1
        if micro % GRAD_ACCUM == 0 or micro == len(order):
            if scaler:
                scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            if scaler:
                scaler.step(opt); scaler.update()
            else:
                opt.step()
            opt.zero_grad(set_to_none=True); sched.step()
            state['step'] += 1; state['micro'] = micro
            if run_loss:
                state['losses'].append(round(float(np.mean(run_loss)), 4)); run_loss = []
            if state['step'] % 10 == 0 or state['step'] == total_steps:
                log(f'train step {state["step"]}/{total_steps} | loss {state["losses"][-1] if state["losses"] else float("nan"):.3f} '
                    f'| {(time.time() - t0) / max(1, state["step"]):.1f} s per step (this session)')
            if micro < len(order) and (time.time() - last_save > SAVE_EVERY_MIN * 60 or out_of_time()):
                save_checkpoint(model, opt, scaler, sched, state); last_save = time.time()
                if out_of_time():
                    return model, False
    save_checkpoint(model, opt, scaler, sched, state, final=True)
    save_json({'losses': state['losses'], 'steps': state['step'], 'examples': len(data), 'train_hash': TRAIN_HASH,
               'lora_r': LORA_R, 'lr': LR, 'epochs': EPOCHS, 'grad_accum': GRAD_ACCUM, 'max_len': MAX_LEN},
              QL_LOCAL / '07_training_log.json')
    put_on_drive(QL_LOCAL / '07_training_log.json', '07_training_log.json')
    return model, True

def for_generation(model):
    """After training: no gradient checkpointing, cache on, eval mode (generation is ~10x slower otherwise)."""
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass
    base = model.get_base_model() if hasattr(model, 'get_base_model') else model
    base.config.use_cache = True
    return model.eval()

if MOCK:   # local testing only: a tiny random Qwen3 and a word-level tokenizer, trained for real on the CPU
    def make_tiny(texts):
        from tokenizers import Tokenizer, models, pre_tokenizers, trainers
        from transformers import PreTrainedTokenizerFast, Qwen3Config, Qwen3ForCausalLM
        t = Tokenizer(models.WordLevel(unk_token='[UNK]'))
        t.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
        t.train_from_iterator(texts, trainers.WordLevelTrainer(special_tokens=['[UNK]', '<|endoftext|>', '<|im_start|>', '<|im_end|>']))
        tok = PreTrainedTokenizerFast(tokenizer_object=t, unk_token='[UNK]', eos_token='<|im_end|>', pad_token='<|endoftext|>')
        tok.chat_template = ("{% for m in messages %}<|im_start|> {{ m['role'] }} {{ m['content'] }} <|im_end|> {% endfor %}"
                             "{% if add_generation_prompt %}<|im_start|> assistant {% endif %}")
        cfg = Qwen3Config(vocab_size=len(tok), hidden_size=32, intermediate_size=64, num_hidden_layers=2,
                          num_attention_heads=2, num_key_value_heads=1, head_dim=16, max_position_embeddings=4096)
        return Qwen3ForCausalLM(cfg), tok

## 6. Run

Each step is skipped when already done (answers on Drive, final adapter on Drive). The cell stops cleanly after 110 minutes; after a disconnect, Run all again.

In [8]:
RUN_LOG = Run(params={'step': 'qlora', 'train_hash': TRAIN_HASH, 'examples': len(TRAIN_ROWS), 'lora_r': LORA_R,
                      'lr': LR, 'epochs': EPOCHS, 'eval_shards': EVAL_SHARDS, 'limit': LIMIT_QUESTIONS})
LLM = TOK = None
finished_all = False
try:
    need_small = any(todo_of(r) for r in RUNS_BASE + RUNS_LORA) or not ADAPTER_FINAL.exists()
    if need_small:
        if not MOCK:
            free_memory()
            if torch.cuda.memory_allocated() / 1e9 > 1.0:
                raise RuntimeError('GPU memory already in use: Disconnect and delete runtime, then Run all once.')
        LLM, TOK = load_llm(SMALL_SPEC)
        log(f'loaded {SMALL_SPEC["id"]} (4-bit)')
        # (a) untuned 4B
        ok = answer_runs(RUNS_BASE)
        # (b) training, or the finished adapter from Drive
        if ok:
            if ADAPTER_FINAL.exists():
                from peft import PeftModel
                d = unzip_to(ADAPTER_FINAL, QL_LOCAL / 'final_in')
                if MOCK:
                    TINY, TINY_TOK = make_tiny([r['memory'] + ' ' + r['question'] + ' ' + ' '.join(r['options']) + ' ' + target_text(r) for r in TRAIN_ROWS])
                    TUNED = PeftModel.from_pretrained(TINY, str(d))
                else:
                    TUNED = PeftModel.from_pretrained(LLM, str(d))
                log('final adapter loaded from Drive')
                done_training = True
            else:
                if MOCK:
                    TINY, TINY_TOK = make_tiny([r['memory'] + ' ' + r['question'] + ' ' + ' '.join(r['options']) + ' ' + target_text(r)
                                                for r in TRAIN_ROWS] + [SYS, 'Reply as {"answer": "<letter>", "evidence": []}'])
                    TUNED, done_training = train(TINY, TINY_TOK)
                else:
                    TUNED, done_training = train(LLM, TOK)
            # (c) tuned 4B
            if done_training:
                TUNED = for_generation(TUNED)
                if not MOCK:
                    LLM = TUNED
                ok = answer_runs(RUNS_LORA)
                if MOCK:                                 # the mock generator ignores the model; check the tiny one works
                    with torch.no_grad():
                        ids = TINY_TOK('<|im_start|> user', return_tensors='pt')['input_ids']
                        print('tiny tuned model generates:', TUNED.generate(input_ids=ids, max_new_tokens=3).shape)
            else:
                ok = False
    else:
        ok = True
    # (d) 8B reference
    if ok and any(todo_of(r) for r in RUNS_8B):
        for n in ['LLM', 'TOK', 'TUNED', 'TINY']:
            globals().pop(n, None)
        LLM = TOK = None
        free_memory()
        if not MOCK:   # the Colab disk holds about 21 GB: remove the 4B download first
            hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
            shutil.rmtree(hub / ('models--' + SMALL_SPEC['id'].replace('/', '--')), ignore_errors=True)
        LLM, TOK = load_llm(BIG_SPEC)
        log(f'loaded {BIG_SPEC["id"]} (4-bit) for the reference run')
        ok = answer_runs(RUNS_8B)
    finished_all = ok and ADAPTER_FINAL.exists() and not any(todo_of(r) for r in ALL_RUNS)
    if not ok:
        log('time budget reached: Run all again to continue (finished steps are skipped)', 'WARN')
finally:
    sync_preds()

[14:33:59] INFO  run started: 07_qlora-20261018-143344-51c3db


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[15:15:59] INFO  loaded Qwen/Qwen3-8B (4-bit) for the reference run
[15:15:59] INFO  star.M4.A: 190 questions to do
[15:19:41] INFO  uploaded 07_shard0_20261018-151941_00006.zip: 6 files, verified on Drive
[15:27:50] INFO  uploaded 07_shard0_20261018-152750_00003.zip: 3 files, verified on Drive
[15:27:50] INFO  star.M4.A: done | 3.7 s per question | accuracy 0.516


## 7. Progress and results so far

Accuracies are **not final results**: notebook 08 scores everything together with confidence intervals and paired tests (untuned vs tuned 4B on the same questions).

In [9]:
try:
    sync_preds()
except DriveSyncError as e:
    log(str(e), 'ERROR')
rows = []
for run in ALL_RUNS:
    recs = [r for f in (LOCAL_PRED / run['id']).glob('*.jsonl') for r in read_jsonl(f)]
    n_have = sum((MEM_LOCAL / run['format'] / f"{q['key']}.json").exists() for q in EVAL_Q[run['qset']])
    rows.append({'run': run['id'], 'model': run['model_id'], 'answered': len({r['uid'] for r in recs}), 'of': n_have,
                 'accuracy': round(np.mean([r['correct'] for r in recs]), 3) if recs else None,
                 'valid_json': round(np.mean([r['valid_json'] for r in recs]), 3) if recs else None,
                 's_per_q': round(np.mean([r['seconds'] for r in recs]), 2) if recs else None})
T = pd.DataFrame(rows)
print(T.to_string(index=False))
tl = load_json(QL / '07_training_log.json', default=None)
if tl:
    L = tl['losses']
    print(f"\ntraining: {tl['examples']} examples, {tl['steps']} steps, loss {L[0]:.3f} (first step) -> "
          f"{np.mean(L[-5:]):.3f} (mean of last 5 steps)")
else:
    print('\ntraining: not finished yet' + (' (a checkpoint is on Drive)' if CKPT.exists() else ''))
left = int((T['of'] - T['answered']).sum())
status = 'done' if (left == 0 and ADAPTER_FINAL.exists()) else 'partial'
print(f'\n{left} answers left' + ('' if status == 'partial' else ': this notebook is finished'))
save_json({'left': left, 'adapter_final': ADAPTER_FINAL.exists(), 'table': rows,
           'updated': dt.datetime.now().isoformat(timespec='seconds')}, P['pred'] / 'progress_07.json')
RUN_LOG.finish(status, notes=f'{left} answers left')

               run               model  answered  of  accuracy  valid_json  s_per_q
     main.M4.A.q4b       Qwen/Qwen3-4B       193 193     0.456       0.876    10.19
     star.M4.A.q4b       Qwen/Qwen3-4B       190 190     0.458       0.932     4.90
main.M4.A.q4b-lora Qwen/Qwen3-4B+qlora       193 193     0.549       1.000     4.23
star.M4.A.q4b-lora Qwen/Qwen3-4B+qlora       190 190     0.537       1.000     2.22
         star.M4.A       Qwen/Qwen3-8B       190 190     0.516       0.989     4.34

training: 900 examples, 113 steps, loss 2.447 (first step) -> 0.069 (mean of last 5 steps)

0 answers left: this notebook is finished
[15:27:52] INFO  run finished: 07_qlora-20261018-143344-51c3db [done] 54.1 min, peak RAM 7.12 GB, peak VRAM 12.61 GB


## What to send back

The table from section 7, and the training line under it. When it says `this notebook is finished`, 07 is done.